In [3]:
#r "nuget: Deedle.interactive"

Installed Packages Deedle.interactive, 3.0.0

Loading extensions from `C:\Users\schne\.nuget\packages\deedle.interactive\3.0.0\lib\netstandard2.1\Deedle.Interactive.dll`

In [6]:
open System.IO
open System.Diagnostics

/// Runs a git command inside `repoPath` and returns (exitCode, stdout, stderr).
let runGit (repoPath: string) (args: string) =
    let psi = ProcessStartInfo(FileName = "git", Arguments = args, WorkingDirectory = repoPath)
    psi.RedirectStandardOutput <- true
    psi.RedirectStandardError <- true
    psi.UseShellExecute <- false
    psi.CreateNoWindow <- true
    use proc = Process.Start(psi)
    let stdout = proc.StandardOutput.ReadToEnd()
    let stderr = proc.StandardError.ReadToEnd()
    proc.WaitForExit()
    proc.ExitCode, stdout, stderr

/// Pulls the latest changes for `repoPath`, but ONLY when the working tree is clean.
/// If there are uncommitted changes (or history has diverged), it refuses to pull
/// and reports the repo as out of sync. Returns true if the repo is now in sync.
let pullIfClean (repoPath: string) =
    // `git status --porcelain` prints one line per change (staged, modified, OR untracked);
    // empty output == clean working tree.
    let _, status, _ = runGit repoPath "status --porcelain"
    if not (System.String.IsNullOrWhiteSpace status) then
        let n = status.Trim().Split('\n').Length
        printfn $"[{repoPath}] OUT OF SYNC - {n} uncommitted change(s) in working tree; skipping pull."
        false
    else
        // Clean tree -> fast-forward only, so diverged history fails loudly instead of auto-merging.
        let code, out, err = runGit repoPath "pull --ff-only"
        if code = 0 then
            printfn $"[{repoPath}] {out.Trim()}"
            true
        else
            printfn $"[{repoPath}] OUT OF SYNC - pull failed: {err.Trim()}"
            false

In [7]:
open Deedle

let df = Frame.ReadCsv("planned_phase_1.csv")   

let base_path = "G:/source/dataplant-gitlab/INSDC_Curation"

let repo_paths =
    df
    |> Frame.mapRows (fun rk os ->
        let datahub_url = os.GetAs<string>("URL")
        let project_accession = os.GetAs<string>("Project_Accession")
        Path.Combine(base_path, project_accession)
    )
    |> Series.values
    |> Seq.toList

let df_with_repos =
    df
    |> fun f ->
        let repo_path_col = 
            f
            |> Frame.mapRows (fun rk os ->
                let datahub_url = os.GetAs<string>("URL")
                let project_accession = os.GetAs<string>("Project_Accession")
                let repo_path = Path.Combine(base_path, project_accession)
                if not (System.IO.Directory.Exists(repo_path)) then
                    printfn $"Cloning {project_accession}..."
                    let git_clone_cmd = $"git clone {datahub_url} {repo_path}"
                    let proc = System.Diagnostics.Process.Start("cmd.exe", $"/C {git_clone_cmd}")
                    proc.WaitForExit()
                    repo_path
                else
                    repo_path
            )
        f
        |> Frame.addCol "Repo_Path" repo_path_col

df_with_repos

0,->,PRJEB25079,6986,ERP106963,https://dee2.io/huge/athaliana/ERP106963_NA.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJEB25079,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079
1,->,PRJNA358059,2516,SRP095347,https://dee2.io/huge/athaliana/SRP095347_GSE92568.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA358059,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059
2,->,PRJNA382136,1407,SRP103736,https://dee2.io/huge/athaliana/SRP103736_GSE97500.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA382136,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136
3,->,PRJNA475542,2359,SRP150217,https://dee2.io/huge/athaliana/SRP150217_GSE115583.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA475542,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542
4,->,PRJNA483458,1732,SRP155742,https://dee2.io/huge/athaliana/SRP155742_GSE117857.zip,True,Kevin,https://git.nfdi4plants.org/insdc_curation/PRJNA483458,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458
:,,...,...,...,...,...,...,...,...,...,...
45,->,PRJNA871888,1114,SRP393237,https://dee2.io/huge/athaliana/SRP393237_GSE211718.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA871888,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA871888
46,->,PRJNA906171,3067,SRP410309,https://dee2.io/huge/athaliana/SRP410309_GSE218944.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA906171,True,True,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA906171
47,->,PRJNA938512,976,SRP424504,https://dee2.io/huge/athaliana/SRP424504_GSE226105.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA938512,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA938512
48,->,PRJNA945404,6909,SRP427656,https://dee2.io/huge/athaliana/SRP427656_GSE227500.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA945404,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA945404
49,->,PRJNA989636,2207,SRP446862,https://dee2.io/huge/athaliana/SRP446862_GSE236290.zip,True,Emre,https://git.nfdi4plants.org/insdc_curation/PRJNA989636,,,G:/source/dataplant-gitlab/INSDC_Curation\PRJNA989636


In [14]:
let pull_all () =
    df_with_repos
    |> Frame.mapRows (fun rk row ->
        let repoPath = row.GetAs<string>("Repo_Path")
        printfn $"Pulling {repoPath}..."
        repoPath, pullIfClean repoPath
    )

In [13]:
let sync_status = pull_all ()

Pulling G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079] Already up to date.
Pulling G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059] Already up to date.
Pulling G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136] Already up to date.
Pulling G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542] Already up to date.
Pulling G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458] Already up to date.
Pulling G:/source/dataplant-gitlab/INSDC_Curation\PRJNA515610...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA515610] Already up to date.
Pulling G:/source/dataplant-gitlab/INSDC_Curation\PRJNA638728...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA638728] Already up to date.
Pulling G:/sour

In [15]:
sync_status
|> Series.filter (fun _ (_,is_synced) -> not is_synced)

(Empty)


In [8]:
let validate_yaml ="""arc_specification: 2.0.0-draft
validation_packages:
  - name: insdc_curation
    version: 0.0.1
"""

let init_validation_yaml (repoPath: string) =
    let wrong_path = Path.Combine(repoPath, ".arc", "validation.yaml")
    // let validation_yaml_path = Path.Combine(repoPath, ".arc", "validation_packages.yml")
    // if not (Directory.Exists(Path.Combine(repoPath, ".arc"))) then
    //     Directory.CreateDirectory(Path.Combine(repoPath, ".arc")) |> ignore
    // if not (File.Exists(validation_yaml_path)) then
    //     File.WriteAllText(validation_yaml_path, validate_yaml)
    //     printfn $"[{repoPath}] Created validation_packages.yml"
    // else
    //     printfn $"[{repoPath}] validation_packages.yml already exists, overwriting"
    //     File.WriteAllText(validation_yaml_path, validate_yaml)
    if File.Exists(wrong_path) then
        printfn $"[{repoPath}] deleting wrong file"
        File.Delete(wrong_path)

    let steps =
        [ 
            "stage",    $"add -u ."
            "commit",   $"commit -m \"remove incorrect file\""
            "push",     $"push" 
        ]

    let rec run remaining =
        match remaining with
        | [] ->
            printfn $"[{repoPath}] done."
        | (name, args) :: rest ->
            let code, _, err = runGit repoPath args
            if code <> 0 then
                printfn $"[{repoPath}] git {name} failed (exit {code}): {err.Trim()}"
            else
                run rest

    run steps

let repo_paths = 
    df_with_repos
    |> Frame.getCol "Repo_Path"
    |> Series.values
    |> Seq.toList

repo_paths
|> List.iteri (fun i repoPath ->
    printfn $"[{i+1}/{repo_paths.Length}] processing {repoPath}..."
    init_validation_yaml repoPath
)


[1/50] processing G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJEB25079] done.
[2/50] processing G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA358059] done.
[3/50] processing G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA382136] done.
[4/50] processing G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA475542] done.
[5/50] processing G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA483458] done.
[6/50] processing G:/source/dataplant-gitlab/INSDC_Curation\PRJNA515610...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA515610] done.
[7/50] processing G:/source/dataplant-gitlab/INSDC_Curation\PRJNA638728...
[G:/source/dataplant-gitlab/INSDC_Curation\PRJNA638728] done.
[8/50] processing G:/source/dataplant-gitla